
# Just uninstall bson and pymongo and only install pymongo ---then restart the kernel/restart the python---dont install bson ever 

# THIS NEW CODE WORKS PERFECTLY GOOD , MODULAR N WITH CONFIGURATIONS

In [0]:
# this code cells works perfectly

# I will start from here ----- s3 ingestion ----- homework from uday bhaiya

# here basically we fetch the data from sql server cloud and dump it in raw data folder of databricks

# we used modular architecture in a sense that we created a common functions such as read n write - which will be called for each file

# also we hide our access key using configuration - we did not hardcode it - we used databricks secret scope - to store access keys

# we also created the widgets to pass path of files

# steps are just previous steps , but this time with the use of common utils functions and confugurations - okay


# =======1.IMPORT LIBRARIES & FUNCTIONS=================

import json
from datetime import date
from common_utils.logging import get_logger
import importlib, common_utils.ingestor
importlib.reload(common_utils.ingestor)
from common_utils.ingestor import read_cosmosdb_json, write_raw

from pymongo import MongoClient
import bson
from bson import json_util

dbutils.library.restartPython()

print("Successfully imported! and kernel restarted successfully")

logger = get_logger('cosmosdb-ingestion')

# =======2.CREATE WIDGETS FOR PATH=====================
dbutils.widgets.text('path',"")
config_path = dbutils.widgets.get('path')

# =======3.CONFIG VARIABLES FOR ACCESSING KEYS USING CONFIG JSON FILE ===========

with open(config_path,'r') as f:
    config = json.load(f)

source_config = config['source']
target_config = config['target']
write_options_target = config['write_options']

print(source_config)


# =======3.CONFIG VARIABLES FOR ACCESSING KEYS USING CONFIG JSON FILE ===========

with open(config_path,'r') as f:
    config = json.load(f)

source_config = config['source']
target_config = config['target']
write_options_target = config['write_options']

print(source_config)

# here no secret keys are there so no need to go to terminal n do scope n access keys etc

# ========4.FETCH DATA FROM SQL SERVER & READ IT USING READ_JDBC FUNCTION=================

df = read_cosmosdb_json(spark,source_config['connection_string'],source_config['database_name'], source_config['connection_name'])
logger.info('read %s rows', df.count())
logger.info('sample data looks like.....')
df.show()

# ========5.WRITING THE DATA INTO RAW DATA FOLDER WITH CURRENT DATE FOLDER ====================

from datetime import date
run_date = date.today().isoformat()
logger.info('load date is %s', run_date)

target_path = f"{target_config["base_path"]}/{target_config["folder"]}/load_date={run_date}"
logger.info("writiing data to %s", target_path)

target_path = write_raw(df, target_path,target_config["file_format"], target_config["mode"], None)
logger.info("data landed at %s", target_path)